In [35]:
import json
from collections import defaultdict
# Setup

%load_ext autoreload
%autoreload 2

import os

import torch
from dotenv import load_dotenv
from pandas import read_json
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoTokenizer
import math

from pipelines.multi_service.gap_statistic import (
    calculate_gap_statistic,
    get_optimal_k_programmatically,
)
from pipelines.multi_service.mean_pooling import mean_pooling

load_dotenv()

print(f"HuggingFace cache directory is {os.environ.get('HF_HOME')}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
HuggingFace cache directory is /media/martin/BigData/datasets/cache/huggingface/


In [24]:
from numpy.random import uniform
from random import sample
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Tokenize inputs

# MODEL = "microsoft/codebert-base"
# MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MODEL = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)

input_texts = [
    "User createUser(User user)",
    "User updateUser(Long userId, User user)",
    "void deleteUser(Long userId)",
    "User getUserById(Long userId)",
    "List<User> listAllUsers()",
    "void activateUser(Long userId)",
    "void deactivateUser(Long userId)",
    "void changeUserPassword(Long userId, String newPassword)",
    "void assignUserRole(Long userId, String role)",
    "void removeUserRole(Long userId, String role)",
    "Potion brewMagicPotion(Ingredient ingredient)",
    "Elixir brewHealingElixir(Ingredient ingredient)",
    "Draught brewStaminaDraught(Ingredient ingredient)",
    "Infusion brewManaInfusion(Ingredient ingredient)",
    "Potion brewFireResistancePotion(Ingredient ingredient)",
    "Serum brewInvisibilitySerum(Ingredient ingredient)",
    "Powder mixExplosivePowder(Ingredient ingredient)",
    # "WeatherReport fetchWeatherUpdate(Location location)",
    # "Gate allocateDepartureGate(Aircraft aircraft)",
    # "Gate releaseDepartureGate(Gate gate)",
    # "Runway reserveRunway(RunwayRequest request)",
    # "Runway releaseRunway(Runway runway)",
    # "LandingReport recordLandingData(LandingData data)",
    # "TakeoffReport recordTakeoffData(TakeoffData data)",
    # "FuelReport calculateFuelConsumption(FlightData data)",
    # "Checklist completeSafetyChecklist(Aircraft aircraft)",
    # "MaintenanceReport performMaintenance(Aircraft aircraft)",
    # "RefuelingTicket refuelAircraft(Aircraft aircraft)",
    # "BoardingPass boardPassenger(Passenger passenger)",
    # "PassengerManifest loadPassengerManifest(Manifest manifest)",
    # "CargoManifest loadCargoManifest(Cargo cargo)",
    # "Plane updateFlightStatus(FlightStatus status)",
    # "Route optimizeFlightRoute(Route route)",
]


# input_texts = [
#     "createUser",
#     "updateUser",
#     "deleteUser",
#     "getUserById",
#     "listAllUsers",
#     "activateUser",
#     "deactivateUser",
#     "changeUserPassword",
#     "assignUserRole",
#     "removeUserRole",
#     "brewMagicPotion",
#     "brewHealingElixir",
#     "brewStaminaDraught",
#     "brewManaInfusion",
#     "brewFireResistancePotion",
#     "brewInvisibilitySerum",
#     "mixExplosivePowder",
#     "fetchWeatherUpdate",
#     "allocateDepartureGate",
#     "releaseDepartureGate",
#     "reserveRunway",
#     "releaseRunway",
#     "recordLandingData",
#     "recordTakeoffData",
#     "calculateFuelConsumption",
#     "completeSafetyChecklist",
#     "performMaintenance",
#     "refuelAircraft",
#     "boardPassenger",
#     "loadPassengerManifest",
#     "loadCargoManifest",
#     "updateFlightStatus",
#     "optimizeFlightRoute",
# ]
#
# input_texts = [
#     "create order",
#     "update order",
#     "delete order",
#     "confirm order",
#     "cancel order",
#     "get order by id",
#     "get orders by customer id",
#     "list all orders",
#     "list open orders",
#     "list closed orders",
#     "calculate order total",
#     "add order item",
#     "remove order item",
#     "update order item quantity",
#     "apply discount to order",
#     "set order status",
#     "mark order as paid",
#     "mark order as shipped",
#     "mark order as delivered",
#     "mark order as returned"
# ]

input_texts = list(
    map(lambda text: "class EmployeeService { " + text + " {...} }", input_texts)
)

tokenized_inputs = tokenizer(
    input_texts, return_tensors="pt", padding=True, truncation=True, max_length=256
)
with torch.no_grad():
    output = model(**tokenized_inputs)


def hopkins(X):
    d = X.shape[1]
    # d = len(vars) # columns
    n = len(X)  # rows
    nbrs = NearestNeighbors(n_neighbors=1).fit(X)

    rand_X = sample(range(0, n, 1), n)

    ujd = []
    wjd = []
    for j in range(0, n):
        u_dist, _ = nbrs.kneighbors(
            uniform(np.amin(X, axis=0), np.amax(X, axis=0), d).reshape(1, -1),
            2,
            return_distance=True,
        )
        ujd.append(u_dist[0][1])
        w_dist, _ = nbrs.kneighbors(
            X[rand_X[j]].reshape(1, -1), 2, return_distance=True
        )
        wjd.append(w_dist[0][1])

    H = sum(ujd) / (sum(ujd) + sum(wjd))
    if math.isnan(H):
        print(ujd, wjd)
        H = 0

    return H


attention_mask = tokenized_inputs["attention_mask"]
# print(token_embeddings.shape)
# print(attention_mask)
# print(attention_mask.sum(axis=-1))
# print(attention_mask.sum(axis=-1).unsqueeze(-1))
# print(token_embeddings.sum(axis=1))
mean_pooled = mean_pooling(output, attention_mask)
# print(mean_pooled)

mean_pooled_numpy = mean_pooled.cpu().numpy()
# print(mean_pooled_numpy)


def entropy(counts):
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


pca = PCA(n_components=0.95, random_state=42)
pca_reduced = pca.fit_transform(mean_pooled_numpy)
print(pca_reduced.shape)

# print(f"Hopkins: {hopkins(pca_reduced)}")
clusters = [2, 3, 4]

for k in clusters:
    kmeans = KMeans(n_clusters=k, random_state=42)
    result = kmeans.fit_predict(pca_reduced)
    print(result)

    # labels_counts = np.zeros(10, dtype=int)
    #
    # for label in result:
    #     labels_counts[label] += 1
    #
    # print(labels_counts)
    # print(entropy(labels_counts))

    # silh_result = silhouette_score(pca_reduced, result)
    #
    # print(f"""K = {k}, silhouette score: {silh_result}""")

gap_df = calculate_gap_statistic(
    pca_reduced, n_refs=5, max_k=math.ceil(math.sqrt(len(input_texts)))
)

# Find the first k where the criterion is met
# We look for the first positive value in 'diff' (or simply max gap if strict rule fails)
print(gap_df.head(8))
gap_result = get_optimal_k_programmatically(gap_df)
print(gap_result)

(17, 11)
[0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1]
[2 2 0 0 2 0 0 0 0 0 1 1 1 1 1 1 1]
[2 2 0 0 2 0 0 0 0 0 1 3 1 1 1 3 1]
   k       gap        sk   gap_k+1    sk_k+1      diff
0  1 -0.150512  0.157427  0.301894  0.089402 -0.363003
1  2  0.301894  0.089402  0.253362  0.070975  0.119507
2  3  0.253362  0.070975  0.250310  0.109217  0.112269
3  4  0.250310  0.109217  0.245250  0.118977  0.124037
4  5  0.245250  0.118977       NaN       NaN       NaN
{'optimal_k': 2, 'is_clusterable': True, 'reason': "1-Standard-Error Rule (First k satisfying Tibshirani's criterion)"}


In [180]:
# Tokenize inputs

MODEL = "microsoft/codebert-base"


# def tokenize_inputs(inputs: list[str]):
#     return map(lambda text_input: tokenizer(text_input, return_tensors="pt"), inputs)

# input_texts = [
#     "UserService[SEP]void getByID()",
#     "UserService[SEP]void getByName()",
#     "UserService[SEP]void ohHello()"
# ]

# input_texts = [
#     "User createUser(User user)",
#     "User updateUser(Long userId, User user)",
#     "void deleteUser(Long userId)",
#     "User getUserById(Long userId)",
#     "List<User> listAllUsers()",
#     "void activateUser(Long userId)",
#     "void deactivateUser(Long userId)",
#     "void changeUserPassword(Long userId, String newPassword)",
#     "void assignUserRole(Long userId, String role)",
#     "void removeUserRole(Long userId, String role)",
#
#     "LaunchSequence launchRocket(Rocket rocket)",
#     "Potion brewMagicPotion(Ingredient ingredient)"
# ]

# input_texts = [
#     "// Method name: createUser",
#     "// Method name: updateUser",
#     "// Method name: deleteUser",
#     "// Method name: getUserById",
#     "// Method name: listAllUsers",
#     "// Method name: activateUser",
#     "// Method name: deactivateUser",
#     "// Method name: changeUserPassword",
#     "// Method name: assignUserRole",
#     "// Method name: removeUserRole",
#     "// Method name: getLaunchedRocket",
#     "// Method name: getBrewedMagicPotion",
#     "UserService"
# ]

# input_texts = [
#     "// Class: EShopService\n// Method: createOrder",
#     "// Class: EShopService\n// Method: confirmOrder",
#     "// Class: EShopService\n// Method: deleteOrder",
#     "// Class: EShopService\n// Method: getOrders",
#     "// Class: EShopService\n// Method: getOrderById",
#     "// Class: EShopService\n// Method: createProduct",
#     "// Class: EShopService\n// Method: getProducts",
#     "// Class: EShopService\n// Method: getProductById",
#     "// Class: EShopService\n// Method: updateProduct",
#     "// Class: EShopService\n// Method: deleteProduct",
#     "// Class: EShopService\n// Method: addCustomer",
#     "// Class: EShopService\n// Method: getCustomers",
#     "// Class: EShopService\n// Method: getCustomerById",
#     "// Class: EShopService\n// Method: removeCustomer",
# ]

# input_texts = [
#     "// Class: OrderService\n// Method: createOrder",
#     "// Class: OrderService\n// Method: updateOrder",
#     "// Class: OrderService\n// Method: deleteOrder",
#     "// Class: OrderService\n// Method: confirmOrder",
#     "// Class: OrderService\n// Method: cancelOrder",
#     "// Class: OrderService\n// Method: getOrderById",
#     "// Class: OrderService\n// Method: getOrdersByCustomerId",
#     "// Class: OrderService\n// Method: listAllOrders",
#     "// Class: OrderService\n// Method: listOpenOrders",
#     "// Class: OrderService\n// Method: listClosedOrders",
#     "// Class: OrderService\n// Method: calculateOrderTotal",
#     "// Class: OrderService\n// Method: addOrderItem",
#     "// Class: OrderService\n// Method: removeOrderItem",
#     "// Class: OrderService\n// Method: updateOrderItemQuantity",
#     "// Class: OrderService\n// Method: applyDiscountToOrder",
#     "// Class: OrderService\n// Method: setOrderStatus",
#     "// Class: OrderService\n// Method: markOrderAsPaid",
#     "// Class: OrderService\n// Method: markOrderAsShipped",
#     "// Class: OrderService\n// Method: markOrderAsDelivered",
#     "// Class: OrderService\n// Method: markOrderAsReturned"
# ]

input_texts = [
    "create order",
    "update order",
    "delete order",
    "confirm order",
    "cancel order",
    "get order by id",
    "get orders by customer id",
    "list all orders",
    "list open orders",
    "list closed orders",
    "calculate order total",
    "add order item",
    "remove order item",
    "update order item quantity",
    "apply discount to order",
    "set order status",
    "mark order as paid",
    "mark order as shipped",
    "mark order as delivered",
    "mark order as returned",
]

# input_texts = list(map(lambda text: "class UserService { " + text + " {...} }", input_texts))


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[
        0
    ]  # First element of model_output contains all token embeddings
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )


# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Tokenize sentences
encoded_input = tokenizer(
    input_texts, padding=True, truncation=True, return_tensors="pt"
)

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

# Normalize embeddings
# sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
mean_pooled_numpy = sentence_embeddings.cpu().numpy()
print("Sentence embeddings:")
print(sentence_embeddings)


# K = math.ceil(math.sqrt(len(input_texts)))
K = 2
kmeans = KMeans(n_clusters=K, random_state=42)
result = kmeans.fit_predict(mean_pooled_numpy)


# def entropy(counts):
#     total = counts.sum()
#     if total == 0:
#         return 0.0
#     p = counts / total
#     p = p[p > 0]
#     return np.sum(p * np.log2(p))


labels_counts = np.zeros(K, dtype=int)

for label in result:
    labels_counts[label] += 1

print(labels_counts)
print(entropy(labels_counts))
# cosine_result = pytorch_cos_sim(mean_pooled_numpy[10], mean_pooled_numpy[14])
# print(cosine_result)

Sentence embeddings:
tensor([[-0.3541, -0.2261,  0.0257,  ...,  0.0177,  0.3455, -0.0453],
        [-0.3325, -0.1847,  0.4260,  ..., -0.3638,  0.0852, -0.2764],
        [-0.3018,  0.2795,  0.2313,  ...,  0.1468,  0.1780, -0.3855],
        ...,
        [-0.4235,  0.0537,  0.3281,  ...,  0.0457,  0.1402, -0.0622],
        [-0.2836,  0.3735,  0.5110,  ..., -0.0335,  0.4223, -0.0301],
        [-0.1840,  0.3176,  0.6899,  ..., -0.2170,  0.0471,  0.0534]])
[ 9 11]
-0.9927744539878083


In [37]:
import pandas as pd

# MODEL = "google-bert/bert-base-uncased"
# MODEL = "google-bert/bert-base-cased"
MODEL = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)

# input_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service.jsonl"
# output_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service-result.jsonl"

input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service.jsonl"
output_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service-result-codebert-signatures.jsonl"

data_frame = read_json(input_file, lines=True)

# origin_property = "methodSignature"
# evaluation_property = "methodSignatures"

evaluation_property = "signature"


def evaluate(inputs: list[str]):
    tokenized_inputs = tokenizer(
        inputs, return_tensors="pt", padding=True, truncation=True, max_length=256
    )
    with torch.no_grad():
        output = model(**tokenized_inputs)

    attention_mask = tokenized_inputs["attention_mask"]
    mean_pooled = mean_pooling(output, attention_mask)

    mean_pooled_numpy = mean_pooled.cpu().numpy()

    pca = PCA(n_components=0.95, random_state=42)
    pca_reduced = pca.fit_transform(mean_pooled_numpy)

    gap_df = calculate_gap_statistic(
        pca_reduced, n_refs=5, max_k=math.ceil(math.sqrt(len(inputs)))
    )

    return (get_optimal_k_programmatically(gap_df)["optimal_k"], pca_reduced)


data_frame["optimal_k"] = pd.Series()
data_frame["cluster"] = pd.Series()
results = []
for index, row in data_frame.iterrows():
    className = row["fullyQualifiedName"]
    methods = row["methods"]

    if len(methods) < 4:
        print(f"Skipped small class {className}, number of methods {len(row)}")
        continue

    if className.endswith("Test"):
        print(f"Skipped Test class {className}, number of methods {len(row)}")
        continue

    def map_m(m):
        return m[evaluation_property]

    mapped_methods = list(map(map_m, methods))
    optimal_k, pca_reduced = evaluate(mapped_methods)

    kmeans = KMeans(n_clusters=optimal_k, random_state=42)
    cluster = kmeans.fit_predict(pca_reduced)
    data_frame.loc[index, "optimal_k"] = optimal_k
    data_frame.at[index, "cluster"] = cluster.tolist()

    results.append(f"{className}: {optimal_k}, {cluster}")

for result in results:
    print(result)

data_frame.to_json(output_file, lines=True, orient="records")

Skipped small class io.aiven.klaw.clusterapi.KafkaClusterApiApplication, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.ClusterTopicRequest, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.ClusterSchemaRequest, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.ApiResponse, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.ServiceAccountDetails, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.SchemaCompatibilityCheckResponse, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.TopicConfig, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.ClusterSchemaCacheResetRequest, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.LoadTopicsResponse, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.config.JwtRequestFilter, number of methods 6
Skipped small class io.aiven.klaw.clusterapi.models.AivenAclRes

In [38]:
input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service-result-codebert-signatures.jsonl"

df = pd.read_json(input_file, lines=True)
one_cluster_lines = df.loc[df.get("optimal_k") == 1]

count = 0
evaluation_property = "methods"
for _, record in one_cluster_lines.iterrows():
    if len(record[evaluation_property]) > 10:
        # print(record["methodSignatures"])
        count += 1

print(count)

more_cluster_lines = df.loc[df.get("optimal_k") > 1]

class_clusters = defaultdict(dict)

for _, record in more_cluster_lines.iterrows():
    k = record["optimal_k"]
    clusters = record["cluster"]
    methods = record["methods"]
    class_name = record["fullyQualifiedName"]
    cohesion = record["lcom4"]

    method_clusters = {}
    for index, cluster in enumerate(clusters):
        cluster_number_str = str(cluster)
        method = methods[index]
        if cluster_number_str in method_clusters:
            method_clusters[cluster_number_str].append(method)
        else:
            method_clusters[cluster_number_str] = [method]

    class_clusters[class_name]["methods"] = method_clusters
    class_clusters[class_name]["lcom4"] = cohesion

for className, metadata in class_clusters.items():
    print(f"--- Class {className}, LCOM4: {metadata['lcom4']} ---")
    for cluster, methods in metadata["methods"].items():
        print(f"Cluster {cluster}: {json.dumps(methods, indent=2)}\n")
    print()

more_cluster_lines

21
--- Class io.aiven.klaw.clusterapi.controller.KafkaConnectController, LCOM4: 1 ---
Cluster 2: [
  {
    "name": "getAllConnectors",
    "signature": "ResponseEntity<ConnectorsStatus> getAllConnectors(String kafkaConnectHost, KafkaSupportedProtocol protocol, String clusterIdentification, boolean connectorStatus)"
  }
]

Cluster 1: [
  {
    "name": "getConnectorDetails",
    "signature": "ResponseEntity<Map<String, Object>> getConnectorDetails(String connectorName, String kafkaConnectHost, KafkaSupportedProtocol protocol, String clusterIdentification)"
  }
]

Cluster 0: [
  {
    "name": "postConnector",
    "signature": "ResponseEntity<ApiResponse> postConnector(ClusterConnectorRequest clusterConnectorRequest)"
  },
  {
    "name": "updateConnector",
    "signature": "ResponseEntity<ApiResponse> updateConnector(ClusterConnectorRequest clusterConnectorRequest)"
  },
  {
    "name": "deleteConnector",
    "signature": "ResponseEntity<ApiResponse> deleteConnector(ClusterConnectorReques

,methods,lcom4,fullyQualifiedName,startLine,optimal_k,cluster
52,"[{'name': 'getAllConnectors', 'signature': 'Re...",1,io.aiven.klaw.clusterapi.controller.KafkaConne...,23,3.0,"[2, 1, 0, 0, 0, 0, 0, 0]"
62,"[{'name': 'createHeaders', 'signature': 'HttpH...",67,io.aiven.klaw.UtilMethods,93,9.0,"[1, 7, 3, 3, 3, 3, 2, 2, 2, 1, 2, 3, 4, 7, 0, ..."
223,"[{'name': 'findById', 'signature': 'Optional<A...",19,io.aiven.klaw.repository.AclRequestsRepo,12,2.0,"[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, ..."
335,"[{'name': 'findById', 'signature': 'Optional<A...",24,io.aiven.klaw.repository.AclRepo,11,2.0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
386,"[{'name': 'findById', 'signature': 'Optional<K...",15,io.aiven.klaw.repository.KwKafkaConnectorReque...,13,3.0,"[2, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1]"
395,"[{'name': 'findById', 'signature': 'Optional<T...",17,io.aiven.klaw.repository.TopicRequestsRepo,14,2.0,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, ..."
411,"[{'name': 'findByUsername', 'signature': 'Opti...",11,io.aiven.klaw.repository.RegisterInfoRepo,8,2.0,"[1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0]"
413,"[{'name': 'findById', 'signature': 'Optional<T...",24,io.aiven.klaw.repository.TopicRepo,11,3.0,"[2, 2, 2, 1, 1, 2, 2, 2, 0, 0, 2, 0, 0, 1, 1, ..."
420,"[{'name': 'getAllTeamsSU', 'signature': 'Respo...",1,io.aiven.klaw.controller.UsersTeamsController,37,3.0,"[0, 0, 0, 1, 1, 2, 2, 2, 2, 0, 1, 1, 2, 2, 2, ..."
421,"[{'name': 'getClusters', 'signature': 'Respons...",1,io.aiven.klaw.controller.EnvsClustersTenantsCo...,30,5.0,"[2, 2, 2, 4, 4, 0, 3, 4, 4, 2, 0, 4, 0, 2, 2, ..."
